# 데이터 불러오기

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay)

df = pd.read_csv('/kaggle/input/datasets/asifxzaman/social-media-addiction-vs-productivity-dataset/social_media_productivity_6000.csv')

df.head()

# EDA

In [ ]:
print(df.shape)

각 피쳐 별로 null이 아닌 데이터 개수, 평균, 표준 편차, 최솟값, 4분위수 출력

addiction_level은 categorical한 데이터라 출력X

In [ ]:
display(df.describe())

In [ ]:
print(df.info())

## Cleaning missing value

In [ ]:
(df.isna().sum() / len(df)) * 100

In [ ]:
print(df.isna().sum())

모든 컬럼에 걸쳐 정확히 120개(2%)씩 결측치가 존재한다. 이는 결측치가 동일한 행에 동시에 발생했을 가능성이 높다.

classification 파트에서는 타겟 변수인 **addiction_level**도 결측치가 있으므로, 결측치가 있는 행 전체를 삭제하는 방식으로 처리한다.

In [ ]:
df = df.dropna()
print(f"결측치 제거 후 Shape: {df.shape}")
print(f"\n결측치 확인:\n{df.isnull().sum()}")

## 데이터 분포

In [ ]:
numerical_cols = df.select_dtypes(include=['float64', 'int64']).columns

plt.figure(figsize=(15, 20))

for i, col in enumerate(numerical_cols):
    plt.subplot(len(numerical_cols) // 2 + 1, 2, i + 1)
    sns.histplot(df[col], kde=True, bins='auto')
    plt.title(f'Distribution of {col}')
    plt.xlabel(col)
    plt.ylabel('Frequency')

plt.tight_layout()
plt.show()

## 클래스 분포 확인

In [ ]:
print('Categorical feature (addiction_level) distribution:')
display(df['addiction_level'].value_counts())

In [ ]:
plt.figure(figsize=(8, 6))
sns.countplot(x='addiction_level', data=df, palette='viridis',
              order=df['addiction_level'].value_counts().index,
              hue='addiction_level', legend=False)
plt.title('Distribution of Addiction Level')
plt.xlabel('Addiction Level')
plt.ylabel('Count')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

Medium이 약 3,184명으로 가장 많고, High 1,857명, Low 959명 순으로 **클래스 불균형**이 존재한다.

특히 Low 클래스가 전체의 약 16%로 가장 적기 때문에, 모델이 Low를 잘 분류하지 못할 가능성이 있다.

이를 고려해 train/test split 시 **stratify** 옵션을 적용하여 각 클래스의 비율이 유지되도록 한다.

## 상관관계 히트맵

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df[numerical_cols].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix of Numerical Features')
plt.tight_layout()
plt.show()

## addiction_level과 다른 피쳐들 간의 boxplot

'addiction_level'을 x축으로 하여 다른 수치형 피쳐들과의 관계를 boxplot으로 시각화

In [ ]:
features_to_plot = [
    'age', 'daily_screen_time', 'social_media_hours', 'study_hours',
    'sleep_hours', 'notifications_per_day', 'focus_score', 'productivity_score'
]

plt.figure(figsize=(15, 20))

for i, feature in enumerate(features_to_plot):
    plt.subplot(len(features_to_plot) // 2 + (len(features_to_plot) % 2), 2, i + 1)
    sns.boxplot(x='addiction_level', y=feature, data=df, palette='viridis',
                order=['Low', 'Medium', 'High'], hue='addiction_level', legend=False)
    plt.title(f'{feature} by Addiction Level')
    plt.xlabel('Addiction Level')
    plt.ylabel(feature)
    plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

boxplot을 통해 addiction_level에 따라 피쳐 분포가 달라지는 경향을 확인할 수 있다.

- **social_media_hours**: addiction level이 높을수록 소셜 미디어 사용 시간이 길어지는 뚜렷한 경향이 있다.
- **daily_screen_time**: addiction level이 높을수록 스크린 타임이 증가한다.
- **notifications_per_day**: High 그룹에서 알림 횟수가 많아지는 경향이 있다.
- **study_hours / sleep_hours**: addiction level이 높을수록 학습 시간과 수면 시간이 줄어드는 경향을 보인다.
- **age / focus_score**: 클래스 간 차이가 상대적으로 작아 분류에 덜 기여할 것으로 예상된다.

# Classification : predict addiction_level

## 타겟 인코딩 (LabelEncoder)

addiction_level은 Low / Medium / High의 3가지 범주형 값을 가지므로 LabelEncoder를 사용해 수치형으로 변환한다.

LabelEncoder는 알파벳순으로 인코딩하므로 High=0, Low=1, Medium=2 순서로 매핑된다.

In [ ]:
le = LabelEncoder()
df['addiction_encoded'] = le.fit_transform(df['addiction_level'])
print(f"클래스 분포:\n{df['addiction_level'].value_counts()}")
print(f"\nLabelEncoder 매핑: {dict(zip(le.classes_, le.transform(le.classes_)))}")

## Feature / Target 분리

productivity_score는 regression 파트의 타겟 변수이므로 feature에서 제외한다.

addiction_level은 인코딩 전 원본 컬럼이므로 함께 제외하고, 인코딩된 addiction_encoded를 타겟으로 사용한다.

In [ ]:
feature_cols = ['age', 'daily_screen_time', 'social_media_hours',
                'study_hours', 'sleep_hours', 'notifications_per_day', 'focus_score']

X = df[feature_cols]
y = df['addiction_encoded']

print(f"Feature shape: {X.shape}")
print(f"Target shape: {y.shape}")

## Scaling

데이터 누수가 발생할 가능성이 있으므로 test data에 대해서는 fit 없이 transform만 적용

KNN과 같은 거리 기반 모델은 feature의 스케일에 민감하므로 StandardScaler를 적용한다.

Decision Tree, Random Forest는 스케일에 영향을 받지 않지만, 모든 모델에 동일한 조건을 적용하기 위해 스케일링을 수행한다.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=feature_cols)

print("=== Feature Scaling 완료 (StandardScaler) ===")
display(X_scaled.describe().round(3))

## Train / Test Split (80:20)

클래스 불균형을 고려해 stratify 옵션을 적용하여 각 클래스의 비율이 train/test에 동일하게 유지되도록 분할한다.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y)

print(f"학습 데이터 개수: {X_train.shape[0]}개")
print(f"검증 데이터 개수: {X_test.shape[0]}개")

## 모델 정의

3가지 분류 모델을 비교한다.

- **Decision Tree**: 직관적인 규칙 기반 분류 모델. 해석이 쉽지만 과적합에 취약하다. max_depth=5로 제한하여 과적합을 방지한다.
- **Random Forest**: 다수의 Decision Tree를 앙상블한 모델. 과적합에 강하고 일반적으로 높은 성능을 보인다.
- **KNN**: 거리 기반 분류 모델. 학습 데이터와 유사한 이웃을 찾아 분류한다. n_neighbors=5로 설정한다.

In [ ]:
models = {
    'Decision Tree':    DecisionTreeClassifier(max_depth=5, random_state=42),
    'Random Forest':    RandomForestClassifier(n_estimators=100, random_state=42),
    'KNN':              KNeighborsClassifier(n_neighbors=5)
}

## K-Fold Cross Validation (k=5) + 테스트셋 평가

StratifiedKFold를 사용해 클래스 비율을 유지하면서 교차 검증을 진행한다.

단순 train/test split만으로 평가하면 데이터 분할에 따라 성능이 달라질 수 있으므로, K-Fold CV로 모델의 안정성과 일반화 성능을 함께 확인한다.

In [ ]:
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

results = {}
print("=== K-Fold Cross Validation (k=5) 결과 ===")
for name, model in models.items():
    cv_scores = cross_val_score(model, X_train, y_train, cv=kfold, scoring='accuracy')
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    test_acc = accuracy_score(y_test, y_pred)
    results[name] = {
        'cv_mean': cv_scores.mean(),
        'cv_std':  cv_scores.std(),
        'test_acc': test_acc,
        'y_pred':  y_pred
    }
    print(f"\n[{name}]")
    print(f"  CV Accuracy: {cv_scores.mean():.4f} \u00b1 {cv_scores.std():.4f}")
    print(f"  Test Accuracy: {test_acc:.4f}")
    print(classification_report(y_test, y_pred, target_names=le.classes_))

3가지 모델 모두 높은 정확도를 보인다. Decision Tree와 Random Forest는 테스트셋 정확도 1.0을 기록하였고, KNN은 0.88의 정확도를 보였다.

classification_report를 통해 각 클래스별 precision, recall, f1-score를 확인할 수 있다.

## 결과 시각화

**[수업에서 다루지 않은 함수 설명]**

`ConfusionMatrixDisplay(confusion_matrix, display_labels)`

sklearn에서 제공하는 Confusion Matrix 시각화 클래스이다. `confusion_matrix` 함수로 계산한 행렬을 인수로 받아 `.plot()` 메서드로 시각화한다.

- `confusion_matrix` : `sklearn.metrics.confusion_matrix()`로 계산된 2D 배열. 행은 실제 클래스, 열은 예측 클래스를 나타낸다.
- `display_labels` : 각 클래스의 레이블 이름. 축에 표시된다.
- `.plot(ax, colorbar, cmap)` : 지정된 matplotlib axes에 히트맵 형태로 행렬을 출력한다. `colorbar`는 색상 바 표시 여부, `cmap`은 색상 맵을 지정한다.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle('Classification Results - Addiction Level', fontsize=16, fontweight='bold')

ax = axes[0, 0]
counts = df['addiction_level'].value_counts()
bars = ax.bar(counts.index, counts.values, color=['#4CAF50','#FF9800','#F44336'])
ax.set_title('Class Distribution (addiction_level)')
ax.set_xlabel('Addiction Level')
ax.set_ylabel('Count')
for bar, val in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
            str(val), ha='center', fontweight='bold')

for idx, (name, res) in enumerate(list(results.items())[:2]):
    ax = axes[0, idx+1]
    cm = confusion_matrix(y_test, res['y_pred'])
    disp = ConfusionMatrixDisplay(cm, display_labels=le.classes_)
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'Confusion Matrix\n{name} (Test Acc: {res["test_acc"]:.4f})')

cm = confusion_matrix(y_test, results['KNN']['y_pred'])
disp = ConfusionMatrixDisplay(cm, display_labels=le.classes_)
disp.plot(ax=axes[1, 0], colorbar=False, cmap='Blues')
axes[1, 0].set_title(f'Confusion Matrix\nKNN (Test Acc: {results["KNN"]["test_acc"]:.4f})')

ax = axes[1, 1]
names = list(results.keys())
means = [results[n]['cv_mean'] for n in names]
stds  = [results[n]['cv_std']  for n in names]
colors = ['#2196F3', '#4CAF50', '#FF9800']
bars = ax.bar(names, means, yerr=stds, color=colors, capsize=8, alpha=0.85)
ax.set_title('K-Fold CV Accuracy Comparison (k=5)')
ax.set_ylabel('Accuracy')
ax.set_ylim(0.5, 1.12)
for bar, mean in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{mean:.4f}', ha='center', fontsize=10, fontweight='bold')

ax = axes[1, 2]
rf_model = models['Random Forest']
importances = pd.Series(rf_model.feature_importances_, index=feature_cols).sort_values(ascending=True)
importances.plot(kind='barh', ax=ax, color='#4CAF50', alpha=0.85)
ax.set_title('Feature Importance (Random Forest)')
ax.set_xlabel('Importance')

plt.tight_layout()
plt.show()

**Confusion Matrix 분석**

Decision Tree와 Random Forest는 테스트셋에서 정확도 1.0을 기록하며 모든 클래스를 완벽하게 분류하였다. KNN은 0.88의 정확도를 보여 세 모델 모두 전반적으로 높은 분류 성능을 나타냈다.

KNN의 경우 일부 클래스 경계 근처의 샘플에서 오분류가 발생하며, 이는 거리 기반 모델의 특성상 클래스 간 경계가 명확하지 않은 영역에서 성능이 다소 낮아지기 때문이다.

**K-Fold CV Accuracy 비교**

Random Forest가 가장 높은 CV 정확도와 가장 작은 표준편차를 보여 안정적이고 일반화 성능이 뛰어남을 확인할 수 있다.

KNN은 Decision Tree보다 높은 정확도를 보이지만, Random Forest에는 미치지 못한다.

**Feature Importance (Random Forest)**

EDA의 boxplot 분석과 일치하게 **social_media_hours**와 **daily_screen_time**이 addiction_level 예측에 가장 중요한 변수로 나타났다.

반면 **age**와 **focus_score**는 상대적으로 중요도가 낮아 클래스 분류에 기여하는 바가 적음을 확인할 수 있다.

## 최종 모델 성능 요약

In [ ]:
summary = pd.DataFrame({
    'Model': list(results.keys()),
    'CV Accuracy (mean)': [f"{results[n]['cv_mean']:.4f}" for n in results],
    'CV Std':             [f"{results[n]['cv_std']:.4f}"  for n in results],
    'Test Accuracy':      [f"{results[n]['test_acc']:.4f}" for n in results]
})
display(summary)

best = max(results, key=lambda n: results[n]['test_acc'])
print(f"\n최고 성능 모델: {best} (Test Acc: {results[best]['test_acc']:.4f})")

세 모델을 비교한 결과 **Decision Tree와 Random Forest**가 테스트셋에서 완벽한 분류 성능을 보였다.

Random Forest는 다수의 Decision Tree를 앙상블하여 개별 트리의 과적합 문제를 완화하고, CV std도 가장 작아 데이터 분할에 관계없이 안정적인 성능을 유지함을 확인할 수 있다.